<a href="https://colab.research.google.com/github/ycchoi60-creator/LangChain/blob/main/Ontology_ToyPalantir.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

필요한 환경변수를 먼저 설정한다.

In [25]:
LLM_PROVIDER = "gemini" # claude 또는 gemini
TEMPERATURE=0.2
KEYFROM_USERDATA = True  # 외부에서 입력하려면 False 로

필요한 모듈을 설치한다.

In [26]:
!pip -q install -U langchain langchain-openai langchain-anthropic langchain-google-genai pandas

API키를 설정한다.

In [27]:
if KEYFROM_USERDATA:
  from google.colab import userdata
  API_KEY = userdata.get('geminiI').strip()
else:
  import getpass
  API_KEY = getpass.getpass('Enter API key: ')

SecretNotFoundError: Secret geminiI does not exist.

### Colab Secrets에 API 키 추가 방법

1.  **왼쪽 패널의 '🔑' 아이콘 클릭**: Colab 화면 왼쪽에 있는 '비밀' 또는 'Secrets' 아이콘(열쇠 모양)을 클릭합니다.
2.  **새 비밀 추가**: '새 비밀 추가' 버튼을 클릭합니다.
3.  **이름 입력**: '이름' 필드에 `geminiI`를 입력합니다. (대소문자 정확히 일치해야 합니다)
4.  **값 입력**: '값' 필드에 실제 Gemini API 키를 붙여넣습니다.
5.  **저장**: '비밀 추가' 또는 '저장' 버튼을 클릭하여 비밀을 저장합니다.
6.  **노트북에서 접근 허용**: 코드 셀 상단에 '액세스' 체크박스가 나타나면 해당 비밀에 대해 액세스를 허용합니다.

실습은 SQLite3를 이용한다. 필요한 모듈 임포트

In [ ]:
import sqlite3
import pandas as pd

LLM을 생성한다.

In [ ]:
if LLM_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(
        model="gpt-5-nano",
        temperature=TEMPERATURE,
        api_key=API_KEY,
      )

elif LLM_PROVIDER == "claude":
    from langchain_anthropic import ChatAnthropic

    llm = ChatAnthropic(
        model="claude-sonnet-4-6",
        temperature=TEMPERATURE,
        api_key=API_KEY,
      )


elif LLM_PROVIDER == "gemini":
    from langchain_google_genai import ChatGoogleGenerativeAI

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=TEMPERATURE,
        api_key=API_KEY,
      )

여기서 부터 실습을 위한 Toy DB를 생성한다.

In [ ]:
conn = sqlite3.connect(":memory:")

가상의 무기 데이터를 테이블을 생성한다.

In [ ]:
conn.execute("""
CREATE TABLE WeaponSystem (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    category TEXT NOT NULL,
    firepower INTEGER NOT NULL,          -- 0~100, 높을수록 화력 큼
    endurance_hours INTEGER NOT NULL,    -- 지속 운용 가능 시간
    range_km INTEGER NOT NULL,           -- 작전 반경 또는 사거리
    mobility INTEGER NOT NULL,           -- 0~100, 높을수록 기동성 좋음
    precision_score INTEGER NOT NULL,    -- 0~100, 높을수록 정밀도 높음
    unit_cost_billion INTEGER NOT NULL,  -- 단가, 억 원 단위
    crew_required INTEGER NOT NULL,      -- 운용 인원
    maintenance_level INTEGER NOT NULL,  -- 1~5, 높을수록 정비 부담 큼
    all_weather INTEGER NOT NULL         -- 1이면 악천후 운용 가능
);
""")

가상의 데이터를 정의한다.

In [ ]:
rows = [
    (1,  "K-Alpha 자주포",      "포병",   92,  8,  40, 55, 72, 80,  5, 4, 1),
    (2,  "SkyEye 정찰드론",     "정찰",   20, 18, 120, 85, 88, 25,  2, 2, 0),
    (3,  "IronWall 방공체계",   "방공",   78, 24,  70, 35, 91, 150, 8, 5, 1),
    (4,  "Thunder 다연장",      "포병",   96,  6,  80, 45, 68, 110, 6, 4, 1),
    (5,  "Falcon 전술드론",     "드론",   65, 14, 100, 90, 84, 35,  2, 2, 0),
    (6,  "Guardian 장갑차",     "기동",   60, 16,  60, 82, 55, 45,  3, 3, 1),
    (7,  "SilentFox 전자전차량", "전자전", 35, 20,  90, 70, 75, 60,  4, 3, 1),
    (8,  "LongBow 미사일체계",  "미사일", 98, 10, 300, 30, 95, 200, 5, 5, 1),
    (9,  "FieldBot 무인지상차량","무인",   45, 22,  50, 78, 70, 30,  1, 2, 0),
    (10, "RapidShield 요격체계", "방공",   82, 12,  65, 60, 93, 130, 4, 4, 1),
    (11, "HawkEye 감시장비",    "감시",   10, 30, 150, 50, 90, 20,  1, 1, 1),
    (12, "Mule 보급차량",       "지원",   25, 28,  80, 88, 40, 18,  2, 1, 1),
]

데이터를 DB에 실제로 입력한다.

In [ ]:
conn.executemany("""
INSERT INTO WeaponSystem
(id, name, category, firepower, endurance_hours, range_km, mobility,
 precision_score, unit_cost_billion, crew_required, maintenance_level, all_weather)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", rows)

DB가 잘 생성되었는지 출력해 보자.

In [ ]:
display(pd.read_sql("SELECT * FROM WeaponSystem", conn))

지금부터 온톨로지를 정의한다. 온톨로지는 LLM에 수동으로 규칙을 알려줘서 규칙기반처럼 작동하게 한다.

In [ ]:
ontology = """
온톨로지 / 의미 규칙:

화력강한체계:
    firepower >= 80

오래가는체계:
    endurance_hours >= 12

장거리체계:
    range_km >= 100

기동성좋은체계:
    mobility >= 75

정밀타격체계:
    precision_score >= 85

저비용체계:
    unit_cost_billion <= 50

소수운용체계:
    crew_required <= 2

정비부담낮은체계:
    maintenance_level <= 2

악천후가능체계:
    all_weather = 1

지속화력체계:
    화력강한체계 AND 오래가는체계
    즉 firepower >= 80 AND endurance_hours >= 12

전방신속투입체계:
    기동성좋은체계 AND 소수운용체계
    즉 mobility >= 75 AND crew_required <= 2

야전운용쉬운체계:
    정비부담낮은체계 AND 소수운용체계
    즉 maintenance_level <= 2 AND crew_required <= 2

가성비정찰체계:
    장거리체계 AND 저비용체계 AND category IN ('정찰', '감시', '드론')
    즉 range_km >= 100 AND unit_cost_billion <= 50 AND category IN ('정찰', '감시', '드론')

고신뢰방어체계:
    정밀타격체계 AND 악천후가능체계 AND category IN ('방공', '미사일')
    즉 precision_score >= 85 AND all_weather = 1 AND category IN ('방공', '미사일')
"""

LLM이 schema를 이해할 수 있게 해 준다. LLM은 사전학습을 통해 유사한 형태를 이미 많이 훈련한 상태

In [ ]:
schema = """
WeaponSystem(
    id INTEGER,
    name TEXT,
    category TEXT,                 -- 포병, 정찰, 방공, 드론, 기동, 전자전, 미사일, 무인, 감시, 지원
    firepower INTEGER,             -- 0~100, 높을수록 화력 큼
    endurance_hours INTEGER,       -- 지속 운용 가능 시간
    range_km INTEGER,              -- 작전 반경 또는 사거리
    mobility INTEGER,              -- 0~100, 높을수록 기동성 좋음
    precision_score INTEGER,       -- 0~100, 높을수록 정밀도 높음
    unit_cost_billion INTEGER,     -- 단가, 억 원 단위
    crew_required INTEGER,         -- 운용 인원
    maintenance_level INTEGER,     -- 1~5, 높을수록 정비 부담 큼
    all_weather INTEGER            -- 1이면 악천후 운용 가능
)
"""

LLM을 호출한다.

In [ ]:
def call_llm(prompt):
    response = llm.invoke(prompt)
    sql = response.content.strip()

    # 혹시 LLM이 ```sql 코드블록으로 감싸서 답할 경우 제거
    sql = sql.replace("```sql", "").replace("```", "").strip()
    sql = sql.rstrip(";")

    return sql

온톨로지 체계가 없는 프롬프트 만들기

In [ ]:
prompt_without_ontology_template = f"""
너는 SQLite SQL 생성기다.

사용자의 한국어 요청을 보고 SQL SELECT 문 하나만 만들어라.

조건:
- WeaponSystem 테이블만 사용한다.
- SELECT 문만 만든다.
- 설명하지 말고 SQL만 출력한다.
- 컬럼은 name, category, firepower, endurance_hours, range_km, mobility,
  precision_score, unit_cost_billion, crew_required, maintenance_level, all_weather를 조회한다.
- SQL 코드블록 마크다운은 쓰지 마라.

테이블 스키마:
{schema}

사용자 요청:
{{question}}
"""

온톨로지 체계가 있는 프롬프트 만들기

In [ ]:
prompt_with_ontology_template = f"""
너는 SQLite SQL 생성기다.

사용자의 한국어 요청을 보고 SQL SELECT 문 하나만 만들어라.

조건:
- WeaponSystem 테이블만 사용한다.
- SELECT 문만 만든다.
- 설명하지 말고 SQL만 출력한다.
- 아래 온톨로지의 의미 규칙을 반드시 우선 적용한다.
- 컬럼은 name, category, firepower, endurance_hours, range_km, mobility,
  precision_score, unit_cost_billion, crew_required, maintenance_level, all_weather를 조회한다.
- SQL 코드블록 마크다운은 쓰지 마라.

테이블 스키마:
{schema}

{ontology}

사용자 요청:
{{question}}
"""

온토롤지 있고 없고를 호출해 비교하는 함수 작성

In [ ]:
def compare_sql_generation(question):

    # --------------------------------------------------------
    # A. 온톨로지 없이 SQL 생성
    # --------------------------------------------------------

    prompt_without_ontology = prompt_without_ontology_template.replace(
        "{question}",
        question
    )

    sql_without = call_llm(prompt_without_ontology)

    print("\n[온톨로지 없음] LLM 생성 SQL")
    print("-" * 100)
    print(sql_without)

    print("\n[온톨로지 없음] SQL 실행 결과")
    print("-" * 100)

    try:
        result_without = pd.read_sql(sql_without, conn)
        display(result_without)
    except Exception as e:
        print("SQL 실행 오류:", e)

    # --------------------------------------------------------
    # B. 온톨로지 포함 SQL 생성
    # --------------------------------------------------------

    prompt_with_ontology = prompt_with_ontology_template.replace(
        "{question}",
        question
    )

    sql_with = call_llm(prompt_with_ontology)

    print("\n[온톨로지 있음] LLM 생성 SQL")
    print("-" * 100)
    print(sql_with)

    print("\n[온톨로지 있음] SQL 실행 결과")
    print("-" * 100)

    try:
        result_with = pd.read_sql(sql_with, conn)
        display(result_with)
    except Exception as e:
        print("SQL 실행 오류:", e)

이제 질문을 하나씩 해 보자.

In [ ]:
question = "화력 세고 오래가는 체계 찾아줘"
compare_sql_generation(question)

In [ ]:
question = "전방에 빨리 넣을 수 있고 운용 인원이 적은 체계 찾아줘"
compare_sql_generation(question)

In [ ]:
question = "정비 부담 낮고 소수 인원으로 운용 가능한 체계 추천해줘"
compare_sql_generation(question)

In [ ]:
question = "장거리 정찰 가능한데 비용이 너무 비싸지 않은 체계 찾아줘"
compare_sql_generation(question)

In [ ]:
question = "악천후에도 쓸 수 있는 정밀한 방어 체계 찾아줘"
compare_sql_generation(question)